# Stack 2 : RAG hybride — BM25 + vectoriel + RRF

Ce notebook construit un RAG **hybride** : il combine la recherche **vectorielle**
(FAISS) et la recherche **lexicale** (BM25), puis fusionne les deux classements
avec **Reciprocal Rank Fusion (RRF)**.

- **Données / Chunking / Embeddings** : identiques au Stack 1 (variables tenues constantes).
- **Nouveauté** : index BM25 en mémoire + fusion RRF.
- **Génération** : Ollama (`llama3.1`).
- Le code de recherche vit dans `src/stack2_hybrid/` — ce notebook l'**importe**, il ne le réécrit pas.

## 1. Configuration

In [ ]:
import sys
import time
import json
from pathlib import Path

import numpy as np

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from shared.chunker import recursive_chunk, Chunk
from shared.embeddings import EmbeddingModel
from shared.llm import call_llm

print(f"Project root: {PROJECT_ROOT}")

## 2. Chargement des données

In [ ]:
from datasets import load_dataset

ds = load_dataset("wikimedia/wikipedia", "20231101.simple", split="train[:100]")
print(f"{len(ds)} articles chargés")

## 3. Découpage (chunking)

Même découpage récursif que le Stack 1.

In [ ]:
all_chunks: list[Chunk] = []
for doc in ds:
    all_chunks.extend(recursive_chunk(
        doc["text"], max_size=500, overlap=50,
        metadata={"title": doc["title"], "url": doc["url"]},
    ))

texts = [c.text for c in all_chunks]
metadata = [c.metadata for c in all_chunks]
print(f"{len(all_chunks)} chunks")

## 4. Embeddings + index FAISS

La partie vectorielle de l'hybride s'appuie sur le même index FAISS que le Stack 1.

In [ ]:
from shared.vector_index import FaissIndexer

emb_model = EmbeddingModel("all-MiniLM-L6-v2")
indexer = FaissIndexer(dimension=emb_model.dimension)
indexer.add(emb_model.encode(texts), texts, metadata)
print(f"Index FAISS : {indexer.size} vecteurs")

## 5. Récupérateur hybride

`HybridRetriever` récupère N candidats par **vectoriel** et N par **BM25**, puis
fusionne les deux classements par **RRF** : `score = Σ 1/(k + rang)`. Un chunk
bien classé par les deux méthodes remonte en tête.

In [ ]:
from stack2_hybrid import HybridRetriever

hybrid = HybridRetriever(indexer, emb_model)

for i, r in enumerate(hybrid.search("Which science studies plants?", k=5), 1):
    print(f"{i}. (RRF={r['score']:.4f}) [{r['metadata'].get('title', '')}] {r['text'][:70]}...")

### Démonstration : là où l'hybride aide

Sur une requête contenant un **terme exact** (nom propre, code…), BM25 rattrape ce que le vectoriel peut diluer.

In [ ]:
from stack1_traditional import VectorRetriever

vector = VectorRetriever(indexer, emb_model)
query = "Apollo 11"  # terme précis : BM25 aide à le retrouver exactement

print("— Vectoriel seul —")
for r in vector.search(query, k=3):
    print(f"  [{r['metadata'].get('title', '')}] {r['text'][:70]}...")

print("\n— Hybride (vectoriel + BM25 + RRF) —")
for r in hybrid.search(query, k=3):
    print(f"  [{r['metadata'].get('title', '')}] {r['text'][:70]}...")

## 6. Pipeline RAG hybride

`HybridRAG` réutilise le squelette RAG partagé : seule la récupération change.

In [ ]:
from stack2_hybrid import HybridRAG

rag = HybridRAG(hybrid, llm_fn=call_llm)
print(rag.prompt_template)

In [ ]:
question = "What happened in April 1912?"
result = rag.query(question, k=5)

print(f"Question : {question}")
print(f"Réponse : {result['answer']}")
print(f"\nRécupération : {result['retrieval_ms']} ms")
print(f"Génération  : {result['generation_ms']} ms")
print(f"Total       : {result['latency_ms']} ms")

In [ ]:
test_queries = [
    "What happened in April 1912?",
    "Which science studies plants?",
    "Who invented the telephone?",
]
for q in test_queries:
    r = rag.query(q)
    print(f"Q : {q}")
    print(f"R : {r['answer']}")
    print(f"   [{r['latency_ms']} ms]\n")

## 7. Évaluation RAGAS

In [ ]:
eval_path = PROJECT_ROOT / "eval" / "questions.json"
if eval_path.exists():
    with open(eval_path, encoding="utf-8") as f:
        eval_data = json.load(f)
else:
    eval_data = [
        {"question": "What happened in April 1912?",
         "ground_truth": "The RMS Titanic sank after hitting an iceberg on April 15, 1912."},
        {"question": "Which science studies plants?",
         "ground_truth": "Botany is the science that studies plants."},
    ]
print(f"{len(eval_data)} questions d'évaluation")

In [ ]:
eval_questions = [e["question"] for e in eval_data]
eval_ground_truths = [e["ground_truth"] for e in eval_data]

eval_answers, eval_contexts = [], []
for q in eval_questions:
    r = rag.query(q)
    eval_answers.append(r["answer"])
    eval_contexts.append([c["text"] for c in r["contexts"]])
print(f"Réponses générées pour {len(eval_questions)} questions.")

In [ ]:
try:
    from shared.evaluator import evaluate_rag
    ragas_results = evaluate_rag(
        questions=eval_questions, answers=eval_answers,
        contexts=eval_contexts, ground_truths=eval_ground_truths,
    )
    print("RAGAS — Stack 2 (Hybride)")
    for m in ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]:
        print(f"  {m}: {ragas_results.get(m, 'N/A')}")
    out = PROJECT_ROOT / "eval" / "hybrid_eval.json"
    with open(out, "w", encoding="utf-8") as f:
        json.dump(ragas_results, f, indent=2, default=str)
    print(f"Résultats enregistrés : {out}")
except Exception as exc:
    print(f"Évaluation RAGAS impossible : {exc}")
    print("(RAGAS peut nécessiter une config supplémentaire, ex. un LLM juge.)")

## Résumé

Le Stack 2 (hybride) combine le **sens** (vectoriel) et les **mots exacts** (BM25)
via RRF. Plus robuste que le vectoriel seul sur les requêtes mixtes, au prix d'un
index supplémentaire.

Seule la **récupération** diffère du Stack 1 — le chunking, les embeddings, le
prompt et le pipeline de génération sont partagés (`src/shared/`).